# Boston Residential Property Values: Data Preparation

This notebook gathers, cleans, and combines two public data sources:

1. **Boston Property Assessment FY2026** from the City of Boston's Analyze Boston data hub (one row per property).
2. **U.S. Census Bureau American Community Survey (ACS) 2020-2024 5-Year Estimates** by ZIP Code Tabulation Area (one row per ZIP code), which adds neighborhood characteristics.

They are linked by ZIP code, and the result is saved as one analysis-ready file, `boston_homes_final.csv`.

In [1]:
import os
import json
import urllib.request

import numpy as np
import pandas as pd

## 1. Obtain the Boston Property Assessment Data

**Source:** City of Boston, Analyze Boston, Property Assessment FY2026
(https://data.boston.gov/dataset/property-assessment/resource/ee73430d-96c0-423e-ad21-c4cfb54c8961)

The full CSV is downloaded in a single request and cached locally, so later runs read the saved copy instead of downloading again.

`thousands=","` converts values like `"822,900"` into numbers, and reading `ZIP_CODE` and `PID` as strings keeps their leading zeros (for example, `02128`).

In [2]:
PROPERTY_URLS = [
    "https://data.boston.gov/datastore/dump/ee73430d-96c0-423e-ad21-c4cfb54c8961",
    ("https://data.boston.gov/dataset/e02c44d2-3c64-459c-8fe2-e1ce5f38a035/"
     "resource/ee73430d-96c0-423e-ad21-c4cfb54c8961/download/"
     "fy2026-property-assessment-data_rev.csv"),
]
PROPERTY_FILE = "boston_property_fy2026.csv"
HEADERS = {"User-Agent": "Mozilla/5.0"}

if not os.path.exists(PROPERTY_FILE):
    for url in PROPERTY_URLS:
        try:
            print("Downloading from", url)
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req) as response, open(PROPERTY_FILE, "wb") as f:
                f.write(response.read())
            print("Download complete")
            break
        except Exception as e:
            print("  Failed:", e)
            if os.path.exists(PROPERTY_FILE):
                os.remove(PROPERTY_FILE)

raw_props = pd.read_csv(PROPERTY_FILE, thousands=",",
                        dtype={"ZIP_CODE": str, "PID": str},
                        low_memory=False)

print(raw_props.shape)
raw_props.head()

(184552, 68)


,_id,PID,CM_ID,GIS_ID,ST_NUM,ST_ALPHA,ST_NUM2,ST_NAME,UNIT_NUM,CITY,...,KITCHEN_STYLE2,KITCHEN_STYLE3,HEAT_TYPE,HEAT_SYSTEM,AC_TYPE,FIREPLACES,ORIENTATION,NUM_PARKING,PROP_VIEW,CORNER_UNIT
0,1,0100001000,NaN,100001000,195.0,NaN,NaN,Lexington ST,NaN,EAST BOSTON,...,S - Semi-Modern,S - Semi-Modern,W - Ht Water/Steam,NaN,N - None,0.0,NaN,3.0,A - Average,NaN
1,2,0100002000,NaN,100002000,197.0,NaN,NaN,Lexington ST,NaN,EAST BOSTON,...,M - Modern,M - Modern,F - Forced Hot Air,NaN,C - Central AC,0.0,NaN,0.0,A - Average,NaN
2,3,0100003000,NaN,100003000,199.0,NaN,NaN,Lexington ST,NaN,EAST BOSTON,...,S - Semi-Modern,S - Semi-Modern,S - Space Heat,NaN,N - None,0.0,NaN,0.0,A - Average,NaN
3,4,0100004000,NaN,100004000,201.0,NaN,NaN,Lexington ST,NaN,EAST BOSTON,...,S - Semi-Modern,S - Semi-Modern,W - Ht Water/Steam,NaN,N - None,0.0,NaN,0.0,A - Average,NaN
4,5,0100005000,NaN,100005000,203.0,NaN,NaN,Lexington ST,NaN,EAST BOSTON,...,N - No Remodeling,S - Semi-Modern,W - Ht Water/Steam,NaN,N - None,0.0,NaN,0.0,A - Average,NaN


## 2. Clean the Property Data

### 2a. Keep only residential homes

The assessment file covers every parcel in Boston, including commercial buildings, tax-exempt properties, and condo parking spaces. Since the focus here is residential values, only these land use (`LU`) codes are kept:

- `R1` single-family, `R2` two-family, `R3` three-family dwellings
- `CD` residential condominium units

The next cell lists every land use type in the file, so the filter can be checked.

In [3]:
raw_props.groupby(["LU", "LU_DESC"]).size().sort_values(ascending=False).head(25)

LU       LU_DESC                
CD       RESIDENTIAL CONDO          74239
R1       SINGLE FAM DWELLING        30439
R2       TWO-FAM DWELLING           16701
R3       THREE-FAM DWELLING         13169
CM       CONDO MAIN                 11139
CP       CONDO PARKING (RES)         8545
RL - RL  RES LAND (Unusable)         4103
R4       APT 4-6 UNITS               2485
RC       RES /COMMERCIAL USE         2323
E        OTHER EXEMPT BLDG           2149
A        APT 7-30 UNITS              1463
E        CITY OF BOSTON              1382
A        SUBSD HOUSING S- 8          1214
RL - RL  RESIDENTIAL LAND             924
E        OTHER PUBLIC LAND            820
CC       RETAIL CONDO                 771
RL - RL  RES ANCILL IMPROVEMT         759
CL       COMMERCIAL LAND              686
RC       COMM MULTI-USE               633
C        RET/WHSL/SERVICE             624
CC       OFFICE CONDO                 590
C        STRIP CTR STORES             526
E        CHARITABLE Organization      409
 

In [4]:
RESIDENTIAL_CODES = ["R1", "R2", "R3", "CD"]

res = raw_props[raw_props["LU"].isin(RESIDENTIAL_CODES)].copy()
res = res[res["LU_DESC"] != "CONDO STORAGE (RES)"]

print("All properties:        ", len(raw_props))
print("Residential properties:", len(res))
res["LU_DESC"].value_counts()

All properties:         184552
Residential properties: 134724


LU_DESC
RESIDENTIAL CONDO       74239
SINGLE FAM DWELLING     30439
TWO-FAM DWELLING        16701
THREE-FAM DWELLING      13169
MULTIPLE BLDGS/1 LOT      174
MOBILE HOME                 2
Name: count, dtype: int64

### 2b. Select the columns needed for the analysis

`LAND_VALUE` and `BLDG_VALUE` are left out on purpose: `TOTAL_VALUE` is calculated from them, so using them as predictors would be data leakage and would make the model look far more accurate than it really is.

In [5]:
keep_cols = ["PID", "TOTAL_VALUE", "LIVING_AREA", "LAND_SF", "BED_RMS", "FULL_BTH",
           "HLF_BTH", "TT_RMS", "YR_BUILT", "YR_REMODEL", "NUM_PARKING", "FIREPLACES",
           "CITY", "ZIP_CODE", "LU", "LU_DESC", "BLDG_TYPE", "OVERALL_COND",
           "AC_TYPE", "HEAT_TYPE", "PROP_VIEW"]

homes = res[keep_cols].copy()
homes.head()

,PID,TOTAL_VALUE,LIVING_AREA,LAND_SF,BED_RMS,FULL_BTH,HLF_BTH,TT_RMS,YR_BUILT,YR_REMODEL,...,FIREPLACES,CITY,ZIP_CODE,LU,LU_DESC,BLDG_TYPE,OVERALL_COND,AC_TYPE,HEAT_TYPE,PROP_VIEW
0,0100001000,822900,2202.0,1150.0,6.0,3.0,0.0,12.0,1900.0,NaN,...,0.0,EAST BOSTON,02128,R3,THREE-FAM DWELLING,RE - Row End,A - Average,N - None,W - Ht Water/Steam,A - Average
1,0100002000,868400,2307.0,1150.0,3.0,3.0,0.0,9.0,1920.0,2000.0,...,0.0,EAST BOSTON,02128,R3,THREE-FAM DWELLING,RM - Row Middle,A - Average,C - Central AC,F - Forced Hot Air,A - Average
2,0100003000,840300,2268.0,1150.0,5.0,3.0,0.0,13.0,1905.0,1985.0,...,0.0,EAST BOSTON,02128,R3,THREE-FAM DWELLING,RM - Row Middle,A - Average,N - None,S - Space Heat,A - Average
3,0100004000,797300,2028.0,1150.0,5.0,3.0,0.0,11.0,1900.0,1991.0,...,0.0,EAST BOSTON,02128,R3,THREE-FAM DWELLING,RM - Row Middle,A - Average,N - None,W - Ht Water/Steam,A - Average
4,0100005000,844400,2546.0,2010.0,6.0,3.0,0.0,13.0,1900.0,1978.0,...,0.0,EAST BOSTON,02128,R3,THREE-FAM DWELLING,RE - Row End,A - Average,N - None,W - Ht Water/Steam,A - Average


### 2c. Fix data types

Numeric columns are converted with `pd.to_numeric(errors="coerce")`, which turns any leftover text into missing values instead of crashing. ZIP codes are padded back to 5 digits in case any lost their leading zero, and text columns are stripped of extra spaces.

In [6]:
num_cols = ["TOTAL_VALUE", "LIVING_AREA", "LAND_SF", "BED_RMS", "FULL_BTH", "HLF_BTH",
           "TT_RMS", "YR_BUILT", "YR_REMODEL", "NUM_PARKING", "FIREPLACES"]

for col in num_cols:
    homes[col] = pd.to_numeric(homes[col].astype(str).str.replace(",", ""),
                                    errors="coerce")

homes["ZIP_CODE"] = homes["ZIP_CODE"].astype(str).str.strip().str.split(".").str[0].str.zfill(5)

text_cols = ["CITY", "LU", "LU_DESC", "BLDG_TYPE", "OVERALL_COND", "AC_TYPE", "HEAT_TYPE", "PROP_VIEW"]
for col in text_cols:
    homes[col] = homes[col].astype(str).str.strip().str.upper()
    homes[col] = homes[col].replace({"NAN": np.nan, "NONE": np.nan, "": np.nan})

homes.dtypes

PID              object
TOTAL_VALUE       int64
LIVING_AREA     float64
LAND_SF         float64
BED_RMS         float64
FULL_BTH        float64
HLF_BTH         float64
TT_RMS          float64
YR_BUILT        float64
YR_REMODEL      float64
NUM_PARKING     float64
FIREPLACES      float64
CITY             object
ZIP_CODE         object
LU               object
LU_DESC          object
BLDG_TYPE        object
OVERALL_COND     object
AC_TYPE          object
HEAT_TYPE        object
PROP_VIEW        object
dtype: object

### 2d. Remove duplicates and impossible values

- Duplicate property IDs are removed.
- Properties with a missing or very small assessed value (under $10,000) are removed, since those are not realistic home values.
- A build year of 0 or before 1700, or a remodel year of 0, is a placeholder rather than a real year, so those are set to missing.
- Living area of 0 and negative counts are set to missing.

In [7]:
before = len(homes)

homes = homes.drop_duplicates(subset="PID")
homes = homes[homes["TOTAL_VALUE"] >= 10_000]

homes.loc[(homes["YR_BUILT"] < 1700) | (homes["YR_BUILT"] > 2026), "YR_BUILT"] = np.nan
homes.loc[(homes["YR_REMODEL"] < 1700) | (homes["YR_REMODEL"] > 2026), "YR_REMODEL"] = np.nan
homes.loc[homes["LIVING_AREA"] <= 0, "LIVING_AREA"] = np.nan

for col in ["BED_RMS", "FULL_BTH", "HLF_BTH", "TT_RMS", "NUM_PARKING", "FIREPLACES"]:
    homes.loc[homes[col] < 0, col] = np.nan

print("Rows removed:", before - len(homes))
print("Rows remaining:", len(homes))

Rows removed: 0
Rows remaining: 134724


### 2e. Create new features

- **`REMODELED`**: about 40% of properties have no remodel year, which most likely means the home was never remodeled rather than that the data is missing. Instead of dropping those rows, this becomes a yes/no (1/0) feature.
- **`AGE`**: years since the home was built (as of 2026), which is easier to interpret than the build year.
- **`TOTAL_BTH`**: full bathrooms plus half of the half bathrooms.

In [8]:
homes["REMODELED"] = homes["YR_REMODEL"].notna().astype(int)
homes["AGE"] = 2026 - homes["YR_BUILT"]
homes["TOTAL_BTH"] = homes["FULL_BTH"] + 0.5 * homes["HLF_BTH"].fillna(0)

homes[["YR_BUILT", "AGE", "YR_REMODEL", "REMODELED", "FULL_BTH", "HLF_BTH", "TOTAL_BTH"]].head()

,YR_BUILT,AGE,YR_REMODEL,REMODELED,FULL_BTH,HLF_BTH,TOTAL_BTH
0,1900.0,126.0,NaN,0,3.0,0.0,3.0
1,1920.0,106.0,2000.0,1,3.0,0.0,3.0
2,1905.0,121.0,1985.0,1,3.0,0.0,3.0
3,1900.0,126.0,1991.0,1,3.0,0.0,3.0
4,1900.0,126.0,1978.0,1,3.0,0.0,3.0


## 3. Obtain the Census (ACS) Neighborhood Data

**Source:** U.S. Census Bureau, 2020-2024 American Community Survey 5-Year Estimates, accessed through the Census Data API (https://api.census.gov) and saved as `acs_2024_zcta.csv`, which must be in the same folder as this notebook. The 5-year estimates are used because they are the most reliable estimates available at the ZIP code level.

Variables requested:

| Census variable | Meaning |
|---|---|
| `B19013_001E` | Median household income |
| `B25003_001E` | Total occupied housing units |
| `B25003_003E` | Renter-occupied housing units |
| `B15003_001E` | Population age 25 and over |
| `B15003_022E` | Population age 25+ with a bachelor's degree |
| `B01003_001E` | Total population |

In [9]:
ACS_FILE = "acs_2024_zcta.csv"
acs_vars = ["B19013_001E", "B25003_001E", "B25003_003E",
            "B15003_001E", "B15003_022E", "B01003_001E"]

acs_raw = pd.read_csv(ACS_FILE, dtype={"zip code tabulation area": str})
print(acs_raw.shape)
acs_raw.head()

(33772, 7)


,B19013_001E,B25003_001E,B25003_003E,B15003_001E,B15003_022E,B01003_001E,zip code tabulation area
0,19454,5768,1877,12511,1788,16669,00601
1,21420,12954,3425,28625,5252,37233,00602
2,20933,20131,8600,36389,6704,48448,00603
3,20992,1860,381,3982,367,5163,00606
4,24496,9604,2725,19399,3224,25357,00610


## 4. Clean the ACS Data

- The ZIP code column is renamed to `ZIP_CODE` to match the property data.
- The Census Bureau uses large negative numbers (such as `-666666666`) when an estimate is suppressed because the sample is too small. These are converted to missing values.
- Counts are turned into percentages so that ZIP codes of different sizes can be compared.
- Only ZIP codes that appear in the Boston property data are kept.

In [10]:
acs = acs_raw.rename(columns={"zip code tabulation area": "ZIP_CODE"}).copy()
acs["ZIP_CODE"] = acs["ZIP_CODE"].str.zfill(5)

for col in acs_vars:
    acs[col] = pd.to_numeric(acs[col], errors="coerce")
    acs.loc[acs[col] < 0, col] = np.nan

acs["MEDIAN_INCOME"] = acs["B19013_001E"]
acs["POPULATION"] = acs["B01003_001E"]
acs["PCT_RENTERS"] = acs["B25003_003E"] / acs["B25003_001E"] * 100
acs["PCT_BACHELORS"] = acs["B15003_022E"] / acs["B15003_001E"] * 100

acs = acs[["ZIP_CODE", "MEDIAN_INCOME", "POPULATION", "PCT_RENTERS", "PCT_BACHELORS"]]

boston_zips = homes["ZIP_CODE"].unique()
acs_boston = acs[acs["ZIP_CODE"].isin(boston_zips)].reset_index(drop=True)

print("Boston ZIP codes in property data:", len(boston_zips))
print("Boston ZIP codes found in ACS:   ", len(acs_boston))
acs_boston.sort_values("MEDIAN_INCOME", ascending=False)

Boston ZIP codes in property data: 34
Boston ZIP codes found in ACS:    33


,ZIP_CODE,MEDIAN_INCOME,POPULATION,PCT_RENTERS,PCT_BACHELORS
1,02108,212928.0,3794.0,49.366447,28.604383
27,02210,172124.0,6971.0,75.217960,45.779967
32,02467,169104.0,23001.0,32.297140,27.899103
17,02127,164387.0,36769.0,54.721576,44.685829
2,02109,158843.0,3612.0,68.297362,42.913257
19,02129,157247.0,20032.0,49.469082,37.415453
31,02458,153520.0,13050.0,38.298699,26.729191
29,02445,139747.0,22560.0,47.942554,28.001742
22,02132,138724.0,26444.0,29.855046,30.746954
3,02110,138438.0,2342.0,30.408472,28.643216


## 5. Merge the Two Data Sources

A left join keeps every property, even if its ZIP code has no Census match. Census ZIP Code Tabulation Areas closely match but do not exactly equal postal ZIP codes, so a few ZIP codes (such as PO box or single-building ZIPs) may not match. The cell after the merge lists any that didn't.

In [11]:
final = homes.merge(acs_boston, on="ZIP_CODE", how="left", validate="many_to_one")

print("Rows before merge:", len(homes))
print("Rows after merge: ", len(final))
final.head()

Rows before merge: 134724
Rows after merge:  134724


,PID,TOTAL_VALUE,LIVING_AREA,LAND_SF,BED_RMS,FULL_BTH,HLF_BTH,TT_RMS,YR_BUILT,YR_REMODEL,...,AC_TYPE,HEAT_TYPE,PROP_VIEW,REMODELED,AGE,TOTAL_BTH,MEDIAN_INCOME,POPULATION,PCT_RENTERS,PCT_BACHELORS
0,0100001000,822900,2202.0,1150.0,6.0,3.0,0.0,12.0,1900.0,NaN,...,N - NONE,W - HT WATER/STEAM,A - AVERAGE,0,126.0,3.0,90106.0,43345.0,71.002481,23.488774
1,0100002000,868400,2307.0,1150.0,3.0,3.0,0.0,9.0,1920.0,2000.0,...,C - CENTRAL AC,F - FORCED HOT AIR,A - AVERAGE,1,106.0,3.0,90106.0,43345.0,71.002481,23.488774
2,0100003000,840300,2268.0,1150.0,5.0,3.0,0.0,13.0,1905.0,1985.0,...,N - NONE,S - SPACE HEAT,A - AVERAGE,1,121.0,3.0,90106.0,43345.0,71.002481,23.488774
3,0100004000,797300,2028.0,1150.0,5.0,3.0,0.0,11.0,1900.0,1991.0,...,N - NONE,W - HT WATER/STEAM,A - AVERAGE,1,126.0,3.0,90106.0,43345.0,71.002481,23.488774
4,0100005000,844400,2546.0,2010.0,6.0,3.0,0.0,13.0,1900.0,1978.0,...,N - NONE,W - HT WATER/STEAM,A - AVERAGE,1,126.0,3.0,90106.0,43345.0,71.002481,23.488774


In [12]:
unmatched = final[final["MEDIAN_INCOME"].isna()]
print("Properties without Census income data (no match or suppressed):", len(unmatched))
unmatched["ZIP_CODE"].value_counts()

Properties without Census income data (no match or suppressed): 1


ZIP_CODE
02146    1
Name: count, dtype: int64

Properties in ZIP codes with no Census data are removed, since the neighborhood features are a key part of the analysis.

In [13]:
final = final.dropna(subset=["MEDIAN_INCOME"]).reset_index(drop=True)
print(final.shape)

(134723, 28)


## 6. Final Data Types and Quality Check

Categorical columns are converted to the `category` type. The missing value summary shows how much data is available for each feature.

Some missing values are expected and are left in place for now. For example, about 43% of properties have no remodel year, which is handled by the `REMODELED` feature. How to handle these will depend on which features each model uses.

In [14]:
cat_cols = ["CITY", "ZIP_CODE", "LU", "LU_DESC", "BLDG_TYPE", "OVERALL_COND",
               "AC_TYPE", "HEAT_TYPE", "PROP_VIEW"]
for col in cat_cols:
    final[col] = final[col].astype("category")

final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134723 entries, 0 to 134722
Data columns (total 28 columns):
 #   Column         Non-Null Count   Dtype   
---  ------         --------------   -----   
 0   PID            134723 non-null  object  
 1   TOTAL_VALUE    134723 non-null  int64   
 2   LIVING_AREA    134720 non-null  float64 
 3   LAND_SF        134722 non-null  float64 
 4   BED_RMS        134721 non-null  float64 
 5   FULL_BTH       134722 non-null  float64 
 6   HLF_BTH        134711 non-null  float64 
 7   TT_RMS         134666 non-null  float64 
 8   YR_BUILT       134622 non-null  float64 
 9   YR_REMODEL     77325 non-null   float64 
 10  NUM_PARKING    134704 non-null  float64 
 11  FIREPLACES     134720 non-null  float64 
 12  CITY           134723 non-null  category
 13  ZIP_CODE       134723 non-null  category
 14  LU             134723 non-null  category
 15  LU_DESC        134723 non-null  category
 16  BLDG_TYPE      134721 non-null  category
 17  OVERALL_CO

In [15]:
missing = pd.DataFrame({
    "missing": final.isna().sum(),
    "percent_missing": (final.isna().mean() * 100).round(1)
}).sort_values("percent_missing", ascending=False)
missing

,missing,percent_missing
YR_REMODEL,57398,42.6
OVERALL_COND,6995,5.2
AGE,101,0.1
YR_BUILT,101,0.1
PID,0,0.0
LU_DESC,0,0.0
PCT_RENTERS,0,0.0
POPULATION,0,0.0
MEDIAN_INCOME,0,0.0
TOTAL_BTH,1,0.0


In [16]:
final.describe().round(1)

,TOTAL_VALUE,LIVING_AREA,LAND_SF,BED_RMS,FULL_BTH,HLF_BTH,TT_RMS,YR_BUILT,YR_REMODEL,NUM_PARKING,FIREPLACES,REMODELED,AGE,TOTAL_BTH,MEDIAN_INCOME,POPULATION,PCT_RENTERS,PCT_BACHELORS
count,134723.0,134720.0,134722.0,134721.0,134722.0,134711.0,134666.0,134622.0,77325.0,134704.0,134720.0,134723.0,134622.0,134722.0,134723.0,134723.0,134723.0,134723.0
mean,926967.0,1684.6,2746.0,3.1,1.7,0.3,6.9,1932.6,2003.2,1.3,0.3,0.6,93.4,1.9,108838.4,31064.7,60.7,30.1
std,880103.6,1057.9,2895.2,2.1,0.9,0.5,4.0,45.1,15.4,1.6,0.7,0.5,45.1,0.9,35040.1,12189.8,15.7,9.1
min,55900.0,130.0,130.0,0.0,0.0,0.0,1.0,1700.0,1900.0,0.0,0.0,0.0,1.0,0.0,43136.0,1292.0,26.6,13.0
25%,542900.0,903.0,901.0,2.0,1.0,0.0,4.0,1900.0,1988.0,0.0,0.0,0.0,62.0,1.0,82953.0,23479.0,54.7,23.5
50%,736900.0,1366.0,1527.0,3.0,2.0,0.0,6.0,1920.0,2006.0,1.0,0.0,1.0,106.0,2.0,98224.0,31151.0,58.6,28.5
75%,1016500.0,2231.4,4152.0,4.0,2.0,1.0,9.0,1964.0,2017.0,2.0,1.0,1.0,126.0,2.5,132210.0,40055.0,73.5,39.4
max,36265800.0,21710.8,234701.0,21.0,21.0,7.0,20.0,2025.0,2026.0,22.0,12.0,1.0,326.0,21.0,212928.0,56216.0,99.2,56.3


## 7. Save the Cleaned Dataset

In [17]:
final.to_csv("boston_homes_final.csv", index=False)
print("Saved boston_homes_final.csv with", final.shape[0], "rows and", final.shape[1], "columns")

Saved boston_homes_final.csv with 134723 rows and 28 columns
